Connecting to databases with ADBC is great, but what do you do when you want to get your data out of one database and into another?

The usual approach is to use your database's built-in export to CSV or some other format, but now you have new problems: this is slow, not necessarily type-safe, and you still have to figure out how to get the export into your other database.

Newer databases like [DuckDB](https://duckdb.org) partially improve this situation with extensions, but it requires a DuckDB-specific extension for your source database, and it only moves your data into DuckDB.

It turns out that ADBC solves this problem very well! If your source and target database have ADBC drivers, you can efficiently move data between them, and you can do it all while never materializing all of the data.

The key aspect of ADBC that makes this efficient is that all your data can move through what's called an Arrow [RecordBatchReader](https://arrow.apache.org/docs/python/generated/pyarrow.RecordBatchReader.html), which is a data structure for reading chunks of a potentially larger-than-memory result set. ADBC drivers can work directly with `RecordBatchReader` when pulling data from one database and inserting into another, and only one chunk of your data needs to be materialized in memory at any given time.

This notebook demonstrates the pattern end-to-end: a Parquet file is loaded into a [Flight SQL](https://arrow.apache.org/docs/format/FlightSql.html) compatible database ([GizmoSQL](https://gizmodata.com/gizmosql)), then streamed into [DuckDB](https://duckdb.org/), all through Apache Arrow's `RecordBatchReader` so the full dataset is never fully materialized in memory. We use GizmoSQL as the Flight SQL server because it's fast and easy to get running locally, but the same pattern will work with any combination of databases.

Requirements:

- Python 3
- Docker

## Setup

Install the required dependencies:

In [ ]:
%pip install -q adbc-driver-manager pyarrow dbc

Install the Flight SQL ADBC driver:

In [ ]:
!dbc install -q flightsql

Install the DuckDB ADBC driver:

In [ ]:
!dbc install -q duckdb

If you don't already have a GizmoSQL instance running, start one with Docker:

In [ ]:
!docker run -d --rm -it --init -p 31337:31337 --name gizmosql \
    -e TLS_ENABLED=0 \
    -e GIZMOSQL_PASSWORD=gizmosql_password \
    --pull always gizmodata/gizmosql:latest-slim

Import the required modules:

In [ ]:
import pyarrow.parquet as pq
from adbc_driver_manager import dbapi

## Load the Parquet File

The next few steps are just setup.

First, load the Parquet file as a PyArrow Table:

In [ ]:
penguins = pq.read_table("../data/penguins/penguins.parquet")

## Connect

Open connections to both GizmoSQL and DuckDB. Keeping them as plain variables (instead of context managers) lets both connections stay open across cells:

In [ ]:
con_gizmosql = dbapi.connect(
    driver="flightsql",
    db_kwargs={
        "uri": "grpc+tcp://localhost:31337",
        "username": "gizmosql_user",
        "password": "gizmosql_password",
    },
)
con_duckdb = dbapi.connect(driver="duckdb")

## Ingest into GizmoSQL

Ingest the table into GizmoSQL:

In [ ]:
cur_gizmosql = con_gizmosql.cursor()
cur_gizmosql.adbc_ingest("penguins", penguins, mode="create")
cur_gizmosql.execute("SELECT count(1) FROM penguins")
print(f"GizmoSQL: {cur_gizmosql.fetchone()[0]:,} rows")

## Transfer to DuckDB

Execute a `SELECT` on GizmoSQL and pipe the result directly into DuckDB via `fetch_record_batch()`. This returns a lazy `RecordBatchReader` that streams rows from GizmoSQL as DuckDB consumes them—nothing is fully materialized:

In [ ]:
cur_gizmosql.execute("SELECT * FROM penguins")
cur_duckdb = con_duckdb.cursor()
cur_duckdb.adbc_ingest("penguins", cur_gizmosql.fetch_record_batch(), mode="create")
cur_duckdb.execute("SELECT count(1) FROM penguins")
print(f"DuckDB: {cur_duckdb.fetchone()[0]:,} rows")

## Takeaways

While the above example moved only 344 rows of data between our two databases, it illustrates an important point that can be extrapolated to data of any scale: ADBC is a fast but also uniform API for not only connecting to databases but also connecting databases together. Without a standard interface like ADBC, every pair of databases you wanted to move data between would need a separate solution. In a world with ADBC, you only need two ADBC drivers and a little bit of Python (or your language of choice) code.

## Cleanup

Close the cursors and connections:

In [ ]:
cur_gizmosql.close()
cur_duckdb.close()
con_gizmosql.close()
con_duckdb.close()

Stop the GizmoSQL container:

In [ ]:
!docker stop gizmosql